The following code retrieves metadata of publications from Dimensions. We cannot provide the data directly. We therefore provide the code so that users can retrieve the data upon successful application of the Dimensions API: https://www.dimensions.ai/dimensions-apis/

In [ ]:
!pip install dimcli

In [ ]:
import os
from time import sleep
from tqdm import tqdm
import pandas as pd
import numpy as np
from collections import defaultdict 

import dimcli
dimcli.login(key="xxx", endpoint="https://app.dimensions.ai/api/dsl/v2") # apply access to a dimensions api
dsl = dimcli.Dsl()

### citation data

In [ ]:
def divide_chunks(l, n):
    for i in range(0, len(l), n): 
        yield l[i:i + n]
            
def query_dimensions(prediction, training):

    pub_ids = list(set(prediction['pub_id'].dropna()).union(training['pub_id'].dropna().astype(int)))
    variables = ["id", "year", "authors", "times_cited"]

    pubs = []
    authors = []

    for pub_list in divide_chunks(pub_ids, 500): 
        ids = ', '.join(['"pub.{}"'.format(x) for x in pub_list])
        query = 'search publications where id in [{}] return publications[{}] limit 1000'.format(ids, '+'.join(variables))
        res = dsl.query(query)

        pubs.append(res.as_dataframe())
        authors.append(res.as_dataframe_authors())
    
    return pd.concat(pubs), pd.concat(authors)

In [ ]:
# samples
prediction = pd.read_csv("https://osf.io/download/h7r5w/")
training = pd.read_csv("https://osf.io/download/b3vm9/")

pubs, authors = query_dimensions(prediction, training)
pubs = pubs.drop(columns = "authors") # number of citations
authors = authors[['pub_id', 'researcher_id']].merge(pubs[['id','year']], how = "left", left_on = "pub_id", right_on = "id").drop(columns = 'id')

### author data

In [ ]:
all_authors = authors[['year','researcher_id']].dropna().groupby("researcher_id", as_index = False).agg("max").apply(tuple, axis=1)

for idx, item in enumerate(all_authors): 
    if idx % 500 == 0 and i != 0:   
        time.sleep(60)
        
    researcher_id, max_year = item
        
    if not researcher_id:
        continue
    variables = ["id", "year", "authors", "times_cited"]
    publications = pd.DataFrame()
    i = 0
    while (i == 0 or (1000 + res['stats']['offset']) < res['stats']['total_count']):
        query = 'search publications where researchers.id = "{}" and (year<{}) return publications[{}] limit 1000 skip {}'.format(researcher_id, max_year,'+'.join(variables), i)
        res = dsl.query(query)
        publications = publications.append(res.as_dataframe())
        i += 1000
        
    if publications.empty:
        continue
        
    publications['n_pub'] = 1
    pubs_by_year = publications.groupby('year', as_index=False)['n_pub'].sum().sort_values("year")
    
    citations_by_year = defaultdict(int)
    for pub_list in divide_chunks(publications["id"], 500):
        citation_ids = ', '.join(['"{}"'.format(x) for x in pub_list])

        query = 'search publications where reference_ids in [{}] return year limit 1000'.format(citation_ids, i)
        res = dsl.query(query)
        for year in res['year']:
            citations_by_year[year['id']] += year['count']
        
        if not res['year']:
            citations_by_year[2021] = 0
            continue
    
    citations_by_year = pd.DataFrame.from_dict(citations_by_year, orient = "index", columns = ['n_citation']).reset_index().rename(columns={'index': 'year'})
    citations_by_year = citations_by_year.sort_values("year")
    
    citations_by_year = pd.concat([citations_by_year.head(1), citations_by_year]).reset_index(drop = True)
    citations_by_year.iloc[0]['year'] -= 1
    citations_by_year.iloc[0]['n_citation'] = 0
    
    res = citations_by_year.merge(pubs_by_year, how = "outer", on = 'year').fillna(0)
    
    res['n_citation'] = res['n_citation'].cumsum()
    res['n_pub'] = res['n_pub'].cumsum()
    
    res['researcher_id'] = researcher_id

    res[res['year'] <= max_year].to_csv('author_prev_records.csv', encoding = "utf-8", mode='a', index = False, header = False if os.path.isfile('author_prev_records.csv') else True)

In [ ]:
def first_author_records(authors, author_prev_records):
    for pub_id, researcher_id, year in tqdm(authors.groupby("pub_id").head(1).apply(tuple, axis = 1)):
        res = author_prev_records[(author_prev_records['researcher_id'] == researcher_id) & (author_prev_records['year'] < year)].sort_values('year').tail(1)
        if not researcher_id or res.empty:
            yield {"pub_id": pub_id ,"first_pub": 0, "first_citation": 0}
            continue
        yield {"pub_id": pub_id ,"first_pub": res['n_pub'].iloc[0], "first_citation": res['n_citation'].iloc[0]}
        
def senior_author_records(authors, author_prev_records):
    all_authors = dict([(key, value.set_index('researcher_id').to_dict()['year']) for (key, value) in authors.groupby("pub_id")])
    for pub_id in tqdm(all_authors):
        current = {"pub_id": pub_id ,"senior_pub": 0, "senior_citation": 0}
        for researcher_id in all_authors[pub_id]:
            res = author_prev_records[(author_prev_records['researcher_id'] == researcher_id) & (author_prev_records['year'] < all_authors[pub_id][researcher_id])].sort_values('year').tail(1)
            if res.empty:
                continue
            if res['n_citation'].iloc[0] > current['senior_citation']:
                current['senior_citation'] = res['n_citation'].iloc[0]
                current['senior_pub'] = res['n_pub'].iloc[0]
        yield current

In [ ]:
author_prev_records = pd.read_csv("author_prev_records.csv")
first_authors = pd.DataFrame(first_author_records(authors, author_prev_records))  # first author number of publications and citations
senior_authors = pd.DataFrame(senior_author_records(authors, author_prev_records)) # senior author number of publications and citations